# Cipher Code Kraken - Stage 1: SFT Training

**Purpose:** Supervised fine-tuning on 2000+ creative code examples to teach Gemma 4 31B
the vocabulary and patterns of Awwwards-worthy web development (Three.js, GSAP, Lenis,
WebGL shaders, advanced CSS).

**Model:** unsloth/gemma-4-31B-it (QLoRA 4-bit on A100 40GB)

**Expected runtime:** ~3.5 hours on A100

**Output:** Merged SFT adapter saved to `cipher-sft-merged/` and backed up to Google Drive.
This merged model is the input for Stage 2 (SimPO anti-slop preference optimization).

**W&B project:** cipher-code-kraken / sft-stage

## Cell 1: Environment Setup

Install Unsloth (Colab-optimized build) and W&B for experiment tracking.
You will be prompted to enter your W&B API key on first login.

**Expected output:** Successful pip install, W&B login confirmation.

In [ ]:
# Install Unsloth (Colab-optimized)
!pip install unsloth -q

# Skip W&B — not needed for training, just adds friction
import os
os.environ["WANDB_DISABLED"] = "true"
print("✅ Unsloth installed, W&B disabled (logs to console only)")

## Cell 2: Clone Repo & Load Training Data

Clone the kr8tiv-training repo to get the curated Awwwards training data and configs.
The data was curated from 18 creative dev repos (Three.js, GSAP, Lenis, Codrops, etc.)

**Expected:** 163+ creative code examples in `data/prompts/awwwards-sft.jsonl`

In [ ]:
# Clone the training repo (has all data + configs)
!git clone https://github.com/kr8tiv-ai/kr8tiv-training.git /content/kr8tiv-training 2>/dev/null || echo "Already cloned"

# Symlink data and configs into working directory
!ln -sf /content/kr8tiv-training/data ./data
!ln -sf /content/kr8tiv-training/configs ./configs
!ln -sf /content/kr8tiv-training/scripts ./scripts

# Try to mount Drive for checkpoint backup (optional — training works without it)
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/cipher-models/
    DRIVE_MOUNTED = True
    print("✅ Drive mounted — checkpoints will be backed up")
except:
    print("⚠️ Drive mount skipped — checkpoints saved locally only")

# Verify data exists
!echo "=== Training data ===" && wc -l ./data/prompts/awwwards-sft.jsonl
!echo "=== Config ===" && head -5 ./configs/sft_config.py

## Cell 3: Load Gemma 4 31B with QLoRA

Load the base model in 4-bit quantization via Unsloth's `FastLanguageModel`.
This should use ~18-22GB VRAM on A100 40GB, leaving headroom for training.

**Key settings:**
- `load_in_4bit=True` -- QLoRA quantization, required for A100 40GB
- `max_seq_length=4096` -- Fits most creative code examples
- `dtype=None` -- Auto-detects bf16 on A100

**Watch for:** VRAM usage after load should be ~18-22GB. If higher, reduce `max_seq_length`.

In [ ]:
import torch
from unsloth import FastLanguageModel
import sys
sys.path.append('/content')
from configs.sft_config import *

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=LOAD_IN_4BIT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
)

print(f"Model loaded: {MODEL_ID}")
print(f"VRAM after load: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
print(f"Max seq length: {MAX_SEQ_LENGTH}")

## Cell 4: Apply LoRA Adapter

Apply LoRA (Low-Rank Adaptation) to the model. This freezes the base weights and adds
trainable adapter matrices to attention and MLP layers.

**Key settings:**
- `r=16, alpha=16` -- Moderate rank to prevent catastrophic forgetting (Research Pitfall 2)
- `use_gradient_checkpointing="unsloth"` -- CRITICAL: saves 30% VRAM (Research Pitfall 1)
- All attention + MLP projections targeted for maximum coverage

**Expected:** ~0.5-1% of parameters trainable. This is intentional -- LoRA keeps the
base model's general capabilities intact while learning creative code patterns.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    random_state=RANDOM_STATE,
)

model.print_trainable_parameters()
print(f"VRAM after LoRA: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

## Cell 5: Load and Prepare Dataset

Load the curated Awwwards training data and apply Gemma 4's native chat template.

**Data format:** Each line is a JSON object with:
```json
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

**Expected:** 163+ examples from real creative dev repos (Three.js, GSAP, Lenis, Codrops, etc.)

In [ ]:
from datasets import load_dataset

# Load curated Awwwards data from the cloned repo
SFT_DATA_PATH = "./data/prompts/awwwards-sft.jsonl"
dataset = load_dataset("json", data_files=SFT_DATA_PATH)

# Formatting function for chat template
# Our data uses "messages" field (not "conversations")
def formatting_func(examples):
    messages_list = examples["messages"]
    texts = []
    for messages in messages_list:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

print(f"Dataset size: {len(dataset['train'])} examples")
print(f"Sample text length: {len(dataset['train'][0]['text'])} chars")
print(f"\n--- Sample preview (first 500 chars) ---")
print(dataset['train'][0]['text'][:500])

## Cell 6: Configure and Run SFT Training

Train with SFTTrainer from TRL. Key hyperparameters from research:
- `batch_size=1` + `grad_accum=4` = effective batch size 4 (Pitfall 1: prevents OOM)
- `lr=2e-4` -- Standard for QLoRA SFT
- `epochs=2` -- Enough to learn patterns without overfitting
- `bf16=True` -- A100 native bfloat16 (NOT fp16)
- `logging_steps=10` -- Watch loss curve in W&B for convergence

**Expected behavior:**
- Loss should start ~2.5-3.5 and decrease to ~0.8-1.2 over 2 epochs
- VRAM should peak at ~32-38GB during training
- If VRAM exceeds 39GB, training may OOM -- reduce `MAX_SEQ_LENGTH` to 2048

**Runtime:** ~3.5 hours on A100 for 2000 examples x 2 epochs

In [ ]:
from trl import SFTTrainer, SFTConfig
import os

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    bf16=BF16,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to=REPORT_TO,
    run_name=WANDB_RUN_NAME,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    args=sft_config,
)

# Monitor VRAM before training
print(f"Pre-train VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

trainer.train()

print(f"Post-train VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
print("SFT training complete!")

## Cell 7: Merge LoRA Adapter and Save

Merge the trained LoRA adapter back into the base model weights.
This produces a standalone model that can be loaded without PEFT.

**Output:** `cipher-sft-merged/` directory containing the merged model.
This is the input for Stage 2 (SimPO anti-slop training).

**Backup:** Also copies to Google Drive for persistence across Colab sessions.
If Colab disconnects, you can load from Drive in the SimPO notebook.

In [ ]:
# Merge LoRA adapter into base model
model.save_pretrained_merged(MERGED_OUTPUT_DIR, tokenizer)
print(f"Merged model saved to {MERGED_OUTPUT_DIR}")

# Backup to Drive if mounted
import os
if os.path.exists('/content/drive/MyDrive'):
    !mkdir -p /content/drive/MyDrive/cipher-models/
    !cp -r {MERGED_OUTPUT_DIR} /content/drive/MyDrive/cipher-models/
    print("✅ Backup saved to Google Drive")
else:
    print("⚠️ Drive not mounted — model saved locally at", MERGED_OUTPUT_DIR)
    print("Download it manually or re-run with Drive mounted")

## Cell 8: Quick Sanity Check - Generation Test

Test the SFT-trained model on a creative code prompt to verify it has learned
Three.js/GSAP/Lenis patterns. This is NOT a thorough evaluation -- just a sanity check
that the model generates creative code instead of generic template output.

**What to look for:**
- Response should include Three.js imports and scene setup
- Should reference GSAP for easing/transitions
- Code should be substantial (50+ lines), not a 5-line snippet
- Should NOT look like generic ChatGPT output (no "Welcome to" / "Get Started")

**If output looks generic:** SFT may need more epochs or higher quality training data.

In [ ]:
# Test generation on a creative code prompt
FastLanguageModel.for_inference(model)

test_prompt = "Create a Three.js particle system that responds to mouse movement with GSAP-powered easing transitions"

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=2048, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 80)
print("GENERATION TEST - SFT Stage")
print("=" * 80)
print(response)
print("=" * 80)
print(f"Response length: {len(response)} chars")
print(f"Final VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")